In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import kagglehub
# Data manipulation and analysis
import pandas as pd
import numpy as np

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning - preprocessing and evaluation
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# Machine learning - regression models
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
import xgboost as xgb

# Suppress warnings for cleaner output
import warnings
warnings.simplefilter('ignore')

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
# Task 1: Write your code here:

delivery = os.path.join(path, 'Q1_data.csv')
df_cars_delivery_time = pd.read_csv(delivery)


In [ ]:
# Task 2: Write your code here:
print(f"Dataset shape: {df_cars_delivery_time.shape}")
df_cars_delivery_time.head()

In [ ]:
# Task 3: Write your code here:

df_cars_delivery_time.info()

In [ ]:
# Task 4: Write your code here:
df_cars_delivery_time.describe()

In [ ]:
# Task 5: Write your code here:
# Plot the distribution of Delivery_Time

sns.set_style("whitegrid")

plt.figure(figsize=(10, 6))

# Histogram
sns.histplot(df_cars_delivery_time['Delivery_Time'], kde=True, color='blue', bins=30)
plt.title('Distribution of Delivery_Time', fontsize=16)
plt.xlabel('Energy Consumption', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.show()

print("\n📊 Target Variable Statistics:")
print(f"  • Mean: {df_cars_delivery_time['Delivery_Time'].mean():.2f}")
print(f"  • Median: {df_cars_delivery_time['Delivery_Time'].median():.2f}")
print(f"  • Std Dev: {df_cars_delivery_time['Delivery_Time'].std():.2f}")
print(f"  • Range: [{df_cars_delivery_time['Delivery_Time'].min():.2f}, {df_cars_delivery_time['Delivery_Time'].max():.2f}]")

In [ ]:
# Task 1: Write your code here:
df_cars_delivery_time=df_cars_delivery_time.drop('Order_ID',axis=1)

In [ ]:
# Task 2: Write your code here:
# . Do we have missing values?
def check_missing_values(df_cars_delivery_time):
  missing_values = df_cars_delivery_time.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed")

  else:
    print("\nNo Missing Values Found.")

check_missing_values(df_cars_delivery_time)

In [ ]:
df_cars_delivery_time=df_cars_delivery_time.dropna()
df_cars_delivery_time.isnull().sum()

In [ ]:
# Task 3: Write your code here:
#  Do we have duplicate samples?
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_cars_delivery_time)

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import OneHotEncoder

def encode_categorical_features(df, columns):
    df_encoded = df.copy()
    for col in columns:
        # Initialize OneHotEncoder, handling unknown categories with ignore
        # and setting sparse_output to False for dense array output
        one_hot_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
        # Fit and transform the column, then create a DataFrame for the new columns
        encoded_data = one_hot_encoder.fit_transform(df_encoded[[col]])
        encoded_df = pd.DataFrame(encoded_data, columns=one_hot_encoder.get_feature_names_out([col]), index=df_encoded.index)

        # Concatenate the new one-hot encoded columns and drop the original column
        df_encoded = pd.concat([df_encoded, encoded_df], axis=1)
        df_encoded = df_encoded.drop(col, axis=1)

        print(f"Encoded '{col}' into new columns: {list(one_hot_encoder.get_feature_names_out([col]))}")
    return df_encoded

# List of categorical columns to encode
categorical_cols = ['Weather','Traffic_Level','Time_of_Day','Vehicle_Type']

# Apply One-Hot Encoding using the function
df_encoded = encode_categorical_features(df_cars_delivery_time, categorical_cols)

In [ ]:
df_encoded # you must give ma bounse :)

In [ ]:
# Task 5: Write your code here:
#Apply feature scaling for all features (Use StandardScaler)
from sklearn.preprocessing import StandardScaler

def scale_numerical_features(df, numerical_cols):

    scaler = StandardScaler()
    df_scaled = df.copy()
    df_scaled[numerical_cols] = scaler.fit_transform(df_scaled[numerical_cols])
    return df_scaled, scaler

# Identify numerical features to scale (exclude target variable )
numerical_features_to_scale = ['Distance_km','Preparation_Time_min','Courier_Experience_yrs']

# Apply scaling to the selected numerical features using the function
df_scaled, scaler = scale_numerical_features(df_encoded, numerical_features_to_scale)
df_scaled.head()

In [ ]:
# Task 6: Write your code here:
#Check for target imbalance and state if it is imbalanced or not (keep this cell empty if not needed)

def check_target_imbalance(df, target_column):
  print("Target Distribution:")

  df[target_column].hist()  # Yeah you can just do this :)
  plt.show()

check_target_imbalance(df_cars_delivery_time, "Delivery_Time")

In [ ]:
# Task 1: Write your code here:
X=df_scaled.drop('Delivery_Time',axis=1)
y=df_scaled['Delivery_Time']

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, f1_score

n_splits = 5 # K=5 Folds

skf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

re=0.0
for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):

  print(f"\nFold {fold_idx + 1}/{n_splits}")

  # Get the train & test split for this fold
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # Train
  model=RandomForestRegressor()
  model.fit(X_train,y_train)
  y_pred = model.predict(X_test)

  # Mean Absolute Error results
  mae_lr = mean_absolute_error(y_test, y_pred)
  print(mae_lr)
  re+=mae_lr

  #Print the averaged score across all folds
print('the averaged score across all folds')
res=re/5
print(res)

In [ ]:
# Task 1: Write your code here:
# Feature importance
feature_cols = ['Distance_km', 'Preparation_Time_min', 'Courier_Experience_yrs', 'Weather_Clear',
                'Weather_Foggy', 'Weather_Rainy', 'Weather_Snowy', 'Weather_Windy', 'Traffic_Level_High', 'Traffic_Level_Low',
             'Traffic_Level_Medium','Time_of_Day_Afternoon','Time_of_Day_Evening','Time_of_Day_Morning','Time_of_Day_Night','Vehicle_Type_Bike','Vehicle_Type_Car','Vehicle_Type_Scooter']


feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
# 6. Plot histogram of predictions
plt.figure(figsize=(8, 5))
plt.hist(y_pred, bins=20, color='skyblue', edgecolor='black')
plt.title("Histogram of Validation Predictions")
plt.xlabel("Predicted Value")
plt.ylabel("Frequency")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
# Task Bonus: Write your code here: